In [1]:
import pandas as pd
import numpy as np
from pandas import DataFrame

# Read input data
data = pd.read_excel('data/input6.xlsx')
a = data['a']
b = data['b']
c = data['c']
Minimum_Capacity = data['pmin']
Maximum_Capacity = data['pmax']
ramp_up = data['rampup']
ramp_down = data['rampdown']
Unit = data['Unit']

# Equal Incremental Method
Power_Demand = float(input("Masukan Beban (MW) : "))
P = np.zeros(15)
delta_P = np.zeros(15)
Lambda = 0
increment = 0.1

while abs(Power_Demand - np.sum(P)) > 0.00001:
    Lambda += increment
    P = np.zeros(15)
    for i in range(15):
        P[i] = (Lambda - b[i]) / (2 * c[i])
        if P[i] < Minimum_Capacity[i]:
            P[i] = Minimum_Capacity[i]
        elif P[i] > Maximum_Capacity[i]:
            P[i] = Maximum_Capacity[i]
        if i > 0:
            delta_P[i] = P[i] - P[i-1]
            if delta_P[i] > ramp_up[i]:
                P[i] = P[i-1] + ramp_up[i]
            elif delta_P[i] < -ramp_down[i]:
                P[i] = P[i-1] - ramp_down[i]
    delta_P[0] = P[0]
    while np.sum(P) > Power_Demand:
        idx = np.argmax(delta_P)
        P[idx] -= 0.01
        delta_P[idx] = P[idx] - P[idx-1]
    while np.sum(P) < Power_Demand:
        idx = np.argmin(delta_P)
        P[idx] += 0.01
        delta_P[idx] = P[idx] - P[idx-1]

# Calculate Fuel Cost
squaredP = np.multiply(P, P)
Fuel_Cost = np.add(a, np.multiply(b, P) + np.multiply(c, squaredP))
total_Fuel_Cost = np.sum(Fuel_Cost)

# Output Results
output = DataFrame({'Unit ke': Unit,'Power Produced (MW)': P, 'Fuel Cost ($)': Fuel_Cost})
print(output)
print("Total Power Produced: {:.2f} MW".format(np.sum(P)))
print('Total Biaya Minimum: $', total_Fuel_Cost)


    Unit ke  Power Produced (MW)  Fuel Cost ($)
0         1               247.49    3188.993139
1         2               455.00    5262.525575
2         3               130.00    1537.619400
3         4               130.00    1624.399400
4         5               362.50    4427.078281
5         6               460.00    4711.352600
6         7               465.00    5216.455900
7         8                60.01     918.422206
8         9                25.00     454.474375
9        10               160.00    1921.946800
10       11                20.00     412.494400
11       12                80.00    1057.553200
12       13                25.00     553.511875
13       14                15.00     491.264025
14       15                15.00     510.940575
Total Power Produced: 2650.00 MW
Total Biaya Minimum: $ 32289.03175061667
